In [7]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderUnavailable
import overpy
import overpass
import osm2geojson

import time

import pyperclip
import geojson
import shapely.geometry as geometry
from shapely.ops import linemerge, unary_union, polygonize

import matplotlib.pyplot as plt
import matplotlib

import networkx as nx
import pandas as pd
import numpy as np
import pickle

import matplotlib.pyplot as plt
from sklearn.cluster import SpectralClustering
from sklearn.cluster import KMeans
import node2vec

import folium
import json
from shapely.geometry import mapping

In [8]:
#load up APIs
#geolocator = Nominatim(user_agent="[email]")
api = overpy.Overpass(url="https://overpass.private.coffee/api/interpreter")
#api - overpass.API(endpoint="https://overpass.private.coffee/api/interpreter")

In [9]:
bbox = (41.900131,-88.488317,41.953771, -88.300618)

bicycle = True
pedestrian=True
query = "("
if bicycle: 
    query+= "way[highway=cycleway]({{bbox}});"
    query += "way[highway=path][bicycle=designated]({{bbox}});"
    query += "way[highway=path][bicycle=yes]({{bbox}});"
    query += "way[footway=path][bicycle=yes]({{bbox}});"
if pedestrian: 
    query += "way[footway=sidewalk]({{bbox}});"
    query += "way[footway=crossing]({{bbox}});"
    query += "way[footway=path][foot=yes]({{bbox}});"
    query += "way[highway=path][foot=yes]({{bbox}});"
    query += "way[highway=footway]({{bbox}});"
query += f"way[highway=crossing]({{bbox}});"
    
query += """);
(._;>;);
out;"""
print(query)

(way[highway=cycleway]({{bbox}});way[highway=path][bicycle=designated]({{bbox}});way[highway=path][bicycle=yes]({{bbox}});way[footway=path][bicycle=yes]({{bbox}});way[footway=sidewalk]({{bbox}});way[footway=crossing]({{bbox}});way[footway=path][foot=yes]({{bbox}});way[highway=path][foot=yes]({{bbox}});way[highway=footway]({{bbox}});way[highway=crossing]({bbox}););
(._;>;);
out;


In [10]:
def get_api_query(api, bbox, bicycle=True, pedestrian=True):
    if not pedestrian and not bicycle: 
        return
    #bbox = expand_bbox(bbox)
    query = "("
    if bicycle: 
        query+= f"way[highway=cycleway]{bbox};"
        query += f"way[highway=path][bicycle=designated]{bbox};"
        query += f"way[highway=path][bicycle=yes]{bbox};"
        query += f"way[footway=path][bicycle=yes]{bbox};"
    if pedestrian: 
        query += f"way[footway=sidewalk]{bbox};"
        query += f"way[footway=crossing]{bbox};"
        query += f"way[footway=path][foot=yes]{bbox};"
        query += f"way[highway=path][foot=yes]{bbox};"
        query += f"way[highway=footway]{bbox};"
    query += f"way[highway=crossing]{bbox};"
        
    query += """);
    (._;>;);
    out;"""
    result = api.query(query)
    return result

def make_a_bbox(lower, left, interval): 
    return (lower, left, lower+interval, left+interval)

def expand_bbox(bbox, margin=0.02):
    return (bbox[0]-margin, bbox[1]-margin, bbox[2]+margin, bbox[3]+margin)

In [11]:
# Get 
#(41.765781, -88.345292, 41.853299, -88.183294) is area
interval = 0.2
bbox = make_a_bbox(41.7,-88.5,interval)
replace = True
try: 
    _ = result
    del _
    if replace: 
        print(1/0)
except:
    okay = False
    while not okay:
        try:
            result = get_api_query(api, bbox, bicycle=True, pedestrian=True)
            okay = True
        except: 
            print("Server load was too high. Waiting...")
            time.sleep(5)
        

In [12]:
def convert_to_borders(ways):
    lss = [] 
    
    for ii_w,way in enumerate(ways):
        ls_coords = []
    
        for node in way.nodes:
            ls_coords.append((node.lon,node.lat)) 
    
        lss.append(geometry.LineString(ls_coords))
    
    
    merged = linemerge([*lss]) 
    borders = unary_union(merged) # linestrings to a MultiLineString
    polygons = list(polygonize(borders))
    return merged, borders, polygons

def convert_to_dispersed_borders(ways):
    lss = [] 
    
    for ii_w,way in enumerate(ways):
        ls_coords = []
    
        for node in way.nodes:
            ls_coords.append((node.lon,node.lat)) 
    
        lss.append(geometry.LineString(ls_coords))
    
    
    merged = linemerge([*lss]) 
    return merged
    
#See how many ways each node belongs to
def collect_overlap(ways):
    outdict = dict()
    for way in ways: 
        way_nodes = way.nodes
        for node in way_nodes: 
            if node.id not in outdict.keys(): 
                outdict[node.id] = 1
            else: 
                outdict[node.id] += 1
    outdict = [(i, outdict[i]) for i in outdict.keys()]
    outdict = sorted(outdict, key = lambda x: -x[1])
    return outdict

def make_graph(ways):
    G = nx.Graph()
    
    for way in ways: 
        for node_dex in range(len(way.nodes)): 
            node = way.nodes[node_dex]
            node_id = node.id
            if G.has_node(node_id) == False:
                G.add_node(node_id)
            if node_dex > 0: 
                G.add_edge(node_id,way.nodes[node_dex-1].id)
    return G

def make_new_lcc(G, ways):
    cclist = sorted([i for i in nx.connected_components(G)], key = lambda x: -len(x))
    lcc = cclist[0]
    lcc_ways = []
    lcc_nodes = []
    for way in ways: 
        for node in way.nodes: 
            if node.id in lcc: 
                lcc_ways.append(way)
                break
    lcc_node_set = set()
    for way in lcc_ways:
        for node in way.nodes:
            lcc_node_set.add(node.id)
    return pd.Series(lcc_ways), lcc_node_set

In [13]:
merged, borders, polygons = convert_to_borders(result.ways)
G = make_graph(result.ways)

lcc_ways, lcc_nodes = make_new_lcc(G, result.ways)

lcc_merged, lcc_borders, lcc_polygons = convert_to_borders(lcc_ways)

In [14]:
_ = pd.Series([1,2,3,4,5])
_.isin([1]).any()

True

In [15]:
# Serialize your merged network to GeoJSON
def quick_map(lcc_merged):
    bike_geojson = mapping(lcc_merged)
    
    # Illinois center
    m = folium.Map(location=[40.0, -89.2], zoom_start=6, tiles='CartoDB positron')
    
    # Illinois outline
    folium.GeoJson(
        "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
        name="Illinois",
        style_function=lambda f: {
            'fillColor': '#E6F1FB',
            'color': '#185FA5',
            'weight': 1.5,
            'fillOpacity': 0.25
        } if f['properties']['name'] == 'Illinois' else {
            'fillOpacity': 0,
            'color': 'none',
            'weight': 0
        }
    ).add_to(m)
    
    # Bike/pedestrian network
    folium.GeoJson(
        bike_geojson,
        name="Bike network",
        style_function=lambda f: {
            'color': '#1D9E75',
            'weight': 2,
            'opacity': 0.85
        }
    ).add_to(m)
    
    folium.Rectangle(
        bounds=[[bbox[0], bbox[1]], [bbox[2], bbox[3]]],
        color='#E24B4A',
        weight=2,
        fill=False
    ).add_to(m)
    
    # new_bbox = get_new_bbox(bbox, interval, 'northeast')
    
    # folium.Rectangle(
    #     bounds=[[new_bbox[0], new_bbox[1]], [new_bbox[2], new_bbox[3]]],
    #     color='#E24B4A',
    #     weight=2,
    #     fill=False
    # ).add_to(m)
    
    
    # Zoom to the network
    bounds = lcc_merged.bounds  # (minx, miny, maxx, maxy)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    
    folium.LayerControl().add_to(m)
    return m

#quick_map(lcc_merged)  # displays inline in Jupyter

In [16]:
#remember: 
#longitude becomes less negative as it goes east, more as it goes west
#latitude gets more positive as it goes north, less as it goes south


def find_outside_ends(nodes, bbox, lcc_nodes, G): 
    nodes = [node for node in nodes if node.id in lcc_nodes]
    #print(nodes)
    east = [node for node in nodes if node.lon >= bbox[3] and node.lat > bbox[0] and node.lat < bbox[2]]
    print(f"Just to be sure, there's {len(east)} nodes in this, dawg.")
    east = [node for node in east if G.degree(node.id) == 1]
    print(f"Just to be sure, there's {len(east)} nodes in this, dawg.")
    southeast = [node for node in nodes if node.lon >= bbox[3] and node.lat <= bbox[0]]
    southeast = [node for node in southeast if G.degree(node.id) == 1]
    south = [node for node in nodes if node.lon >= bbox[1] and node.lon < bbox[3] and node.lat < bbox[0]]
    south = [node for node in south if G.degree(node.id) == 1]
    southwest = [node for node in nodes if node.lon <= bbox[1] and node.lat <= bbox[0]]
    southwest = [node for node in southwest if G.degree(node.id)]
    west = [node for node in nodes if node.lon <= bbox[1] and node.lat > bbox[0] and node.lat < bbox[2]]
    west = [node for node in west if G.degree(node.id) == 1]
    northwest = [node for node in nodes if node.lon <= bbox[1] and node.lat >= bbox[2]]
    northwest = [node for node in northwest if G.degree(node.id)]
    north = [node for node in nodes if node.lat >= bbox[2] and node.lon < bbox[3] and node.lon > bbox[1]]
    north = [node for node in north if G.degree(node.id) == 1]
    northeast = [node for node in nodes if node.lat >= bbox[2] and node.lon >= bbox[3]]
    northeast = [node for node in northeast if G.degree(node.id)]
    return {'east': east, 'southeast': southeast, 'south':south, 'southwest':southwest, 'west': west,
            'northwest':northwest, 'north':north, 'northeast':northeast}

def get_new_bbox(bbox, interval, direction): 
    bottom = bbox[0]
    left = bbox[1]
    if direction == 'east': 
        return make_a_bbox(bottom, left + interval, interval)
    elif direction == 'southeast': 
        return make_a_bbox(bottom - interval, left + interval, interval)
    elif direction == 'south': 
        return make_a_bbox(bottom - interval, left, interval)
    elif direction == 'southwest': 
        return make_a_bbox(bottom - interval, left - interval, interval)
    elif direction == 'west': 
        return make_a_bbox(bottom, left - interval, interval)
    elif direction == 'northwest': 
        return make_a_bbox(bottom + interval, left - interval, interval)
    elif direction == 'north':
        return make_a_bbox(bottom + interval, left, interval)
    else: 
        return make_a_bbox(bottom+interval, left + interval, interval)

def find_connecting_bboxes(nodes, bbox, lcc_nodes, G, interval): 
    outside_ends = find_outside_ends(nodes, bbox, lcc_nodes, G)
    outdict = dict()
    for key in outside_ends.keys(): 
        if outside_ends[key] != []: 
            outdict[key] = get_new_bbox(bbox, interval, key)
    return outdict

In [17]:
#creating a list of used bboxes to ensure we don't backtrack


#assembling the nodes and ways we already have

replace = True
if replace: 
    used_bboxes = [bbox]
    total_result_nodes = result.nodes
    total_result_ways = result.ways
    
    print("Assembling graph...")
    G = make_graph(total_result_ways)
    print("Assembling lcc...")
    lcc_ways, lcc_nodes = make_new_lcc(G, total_result_ways)
    print("Making borders...")
    lcc_merged, lcc_borders, lcc_polygons = convert_to_borders(lcc_ways)

    print(f"Before we began, lcc_ways was {len(lcc_ways)} long")
    
    # finding the connecting bboxes
    bbox_dict = find_connecting_bboxes(total_result_nodes, bbox, lcc_nodes, G, interval)
    
    outbox = [bbox_dict[key] for key in bbox_dict.keys()]
    
    print(bbox_dict)
    
boxcount = 1
box_limit = 30

save_bbox_dict = dict()

#iterate through directions and get their readings
while outbox != [] and boxcount <= box_limit: 
    print("I'm starting the new loop")
    #skip used bboxes
    use_bbox = outbox[0]
    if use_bbox in used_bboxes: 
        outbox.pop(0)
        continue
    #print("I'm going through " + key)
    print("I'm going through " + str(use_bbox))
    #try to get the api query until the request goes through
    accepted = False
    bad = 1
    while not accepted:
        try:
            use_result = get_api_query(api, use_bbox, bicycle=True, pedestrian=True)
            save_bbox_dict[use_bbox] = use_result
            accepted = True
            
        except: 
            bad += 1
            print("That didn't work, let's try that again")
            if bad == 10:
                break
            time.sleep(10)
    print("Finished with the query")
    #add our new nodes and ways, then clear out duplicates
    total_result_nodes += use_result.nodes
    total_result_ways += use_result.ways

    node_dict = {n.id: n for n in total_result_nodes}
    way_dict = {w.id: w for w in total_result_ways}
    total_result_nodes = list(node_dict.values())
    total_result_ways = list(way_dict.values())
    
    G = make_graph(total_result_ways)
    
    lcc_ways, lcc_nodes = make_new_lcc(G, total_result_ways)
    print(f"Finished {use_bbox}")
    print(f"lcc_ways is now {len(lcc_ways)} long") 
    used_bboxes.append(use_bbox)
    outbox.pop(0)
    boxcount += 1
    #lcc_merged, lcc_borders, lcc_polygons = convert_to_borders(lcc_ways)
    #print("Assembled the lcc merged/borders")
    new_bboxes = find_connecting_bboxes(total_result_nodes, use_bbox, lcc_nodes, G, interval)
    new_bboxes = [new_bboxes[key] for key in new_bboxes.keys()][::-1]
    print(f"I'm adding {len(new_bboxes)} new bboxes")
    outbox += new_bboxes
   
    quick_map(lcc_merged)
    print("I'm going on to the next loop! See you there!")

Assembling graph...
Assembling lcc...
Making borders...
Before we began, lcc_ways was 12349 long
Just to be sure, there's 1145 nodes in this, dawg.
Just to be sure, there's 92 nodes in this, dawg.
{'east': (41.7, -88.3, 41.900000000000006, -88.1), 'south': (41.5, -88.5, 41.7, -88.3), 'north': (41.900000000000006, -88.5, 42.10000000000001, -88.3)}
I'm starting the new loop
I'm going through (41.7, -88.3, 41.900000000000006, -88.1)
Finished with the query
Finished (41.7, -88.3, 41.900000000000006, -88.1)
lcc_ways is now 48121 long
Just to be sure, there's 972 nodes in this, dawg.
Just to be sure, there's 69 nodes in this, dawg.
I'm adding 6 new bboxes
I'm going on to the next loop! See you there!
I'm starting the new loop
I'm going through (41.5, -88.5, 41.7, -88.3)
That didn't work, let's try that again
Finished with the query
Finished (41.5, -88.5, 41.7, -88.3)
lcc_ways is now 56327 long
Just to be sure, there's 1693 nodes in this, dawg.
Just to be sure, there's 119 nodes in this, dawg

In [18]:
find_connecting_bboxes

<function __main__.find_connecting_bboxes(nodes, bbox, lcc_nodes, G, interval)>

In [19]:
import sys
sys.getsizeof(total_result_ways)

3792656

In [20]:
boxcount

31

In [21]:
# After the loop, rebuild:
lcc_merged, lcc_borders, lcc_polygons = convert_to_borders(lcc_ways)
bike_geojson = mapping(lcc_merged)



In [22]:
total_merged, total_borders, total_polygons = convert_to_borders(total_result_ways)
non_lcc_merged = total_merged - lcc_merged


In [23]:
total_merged = total_merged.simplify(tolerance=0.0001, preserve_topology=True)
non_lcc_merged = non_lcc_merged.simplify(tolerance=0.0001, preserve_topology=True)
lcc_merged = lcc_merged.simplify(tolerance=0.0001, preserve_topology=True)

total_geojson = mapping(total_merged)
non_lcc_geojson = mapping(non_lcc_merged)

In [21]:
overwrite = True
if overwrite:
    with open('05-18-26_chicagoland_bike.geojson', 'w') as file:
        geojson.dump(bike_geojson, file)
    with open('05-18-26_chicagoland_total.geojson', 'w') as file:
        geojson.dump(total_geojson, file)
    with open('05-18-26_chicagoland_non-lcc.geojson', 'w') as file:
        geojson.dump(non_lcc_geojson, file)
    pickle.dump(save_bbox_dict, open('05-18-26_save_bbox_dict.pkl', 'wb'))
    del save_bbox_dict

In [4]:
import geojson
with open('05-18-26_chicagoland_bike.geojson', 'r') as file:
    bike_geojson = geojson.load(file)
with open('05-18-26_chicagoland_total.geojson', 'r') as file:
    total_geojson = geojson.load(file)
with open('05-18-26_chicagoland_non-lcc.geojson', 'r') as file:
    non_lcc_geojson = geojson.load(file)

In [24]:
import folium
import json
from shapely.geometry import mapping

# Serialize merged network to GeoJSON
try:
    bike_geojson = mapping(lcc_merged)
except: 
    bike_geojson=bike_geojson


# Illinois center
m = folium.Map(location=[40.0, -89.2], zoom_start=6, tiles='CartoDB positron')

# Illinois outline
folium.GeoJson(
    "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
    name="Illinois",
    style_function=lambda f: {
        'fillColor': '#E6F1FB',
        'color': '#185FA5',
        'weight': 1.5,
        'fillOpacity': 0.25
    } if f['properties']['name'] == 'Illinois' else {
        'fillOpacity': 0,
        'color': 'none',
        'weight': 0
    }
).add_to(m)

# Bike/pedestrian network
folium.GeoJson(
    bike_geojson,
    name="Bike network",
    style_function=lambda f: {
        'color': '#1D9E75',
        'weight': 2,
        'opacity': 0.85
    }
).add_to(m)

#Non-bike/pedestrian network

# folium.GeoJson(
#     non_lcc_geojson,
#     name="non-connected trails and sidewalks",
#     style_function=lambda f: {
#         'color': '#F50AA9',
#         'weight': 2,
#         'opacity': 0.85
#     }
# ).add_to(m)

# try:
#     for temp_bbox in used_bboxes:
#         folium.Rectangle(
#             bounds=[[temp_bbox[0], temp_bbox[1]], [temp_bbox[2], temp_bbox[3]]],
#             color='#E24B4A',
#             weight=2,
#             fill=False
#         ).add_to(m)
# except:
#     print("no used_bboxes found")

# new_bbox = get_new_bbox(bbox, interval, 'northeast')

# folium.Rectangle(
#     bounds=[[new_bbox[0], new_bbox[1]], [new_bbox[2], new_bbox[3]]],
#     color='#E24B4A',
#     weight=2,
#     fill=False
# ).add_to(m)


# Zoom to the network
try:
    bounds = lcc_merged.bounds  # (minx, miny, maxx, maxy)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
except: 
    print("No bounds found")

folium.LayerControl().add_to(m)

#m  # displays inline in Jupyter

# Louvain

Gives extremely granular communities; interesting, but not especially useful

In [20]:
lcc_subgraph = G.subgraph(lcc_nodes)
communities = nx.community.louvain_communities(lcc_subgraph, seed=42)
communities = sorted(communities, key = lambda x: -len(x))

In [21]:
# Go through each way
# check each node: what community does it belong to?
# If 2 or more nodes belong to the same community, add that way to the output community. 

community_geojson_dict = dict()

for lcc_way_dex in range(len(lcc_ways)):
    #print(f"I am on {lcc_way_dex}/{len(lcc_ways)}")
    lcc_way = lcc_ways[lcc_way_dex]
    count_community_dict = dict() 
    for node in lcc_way.nodes: 
        for community_dex in range(len(communities)):
            use_community = communities[community_dex]
            if node.id in use_community: 
                if community_dex not in count_community_dict.keys(): 
                    count_community_dict[community_dex] = 1
                else: 
                    count_community_dict[community_dex] += 1
                break
        if count_community_dict[community_dex] >= 2: 
            if community_dex not in community_geojson_dict.keys(): 
                community_geojson_dict[community_dex] = [lcc_way]
            else: 
                community_geojson_dict[community_dex].append(lcc_way)
            break


In [2]:
import folium
import json
from shapely.geometry import mapping

# Serialize merged network to GeoJSON

# Illinois center
m = folium.Map(location=[40.0, -89.2], zoom_start=6, tiles='CartoDB positron')

# Illinois outline
folium.GeoJson(
    "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
    name="Illinois",
    style_function=lambda f: {
        'fillColor': '#E6F1FB',
        'color': '#185FA5',
        'weight': 1.5,
        'fillOpacity': 0.25
    } if f['properties']['name'] == 'Illinois' else {
        'fillOpacity': 0,
        'color': 'none',
        'weight': 0
    }
).add_to(m)

# Bike/pedestrian network

color_palette = ["#0adfcc",'#dba9cf','#0375a6',"#bc2f83", "#6cb04d"]

for map_community_double_dex in range(len(list(community_geojson_dict.keys()))):
    map_community_dex = list(community_geojson_dict.keys())[map_community_double_dex]
    use_map_community = community_geojson_dict[map_community_dex]
    use_map_community_merged = convert_to_dispersed_borders(use_map_community)
    #use_map_community_merged = use_map_community_merged.simplify(tolerance=0.0001, preserve_topology=True)
    um_bike_geojson = mapping(use_map_community_merged)
    test_color_palette = color_palette[map_community_double_dex % len(color_palette)]
    print(test_color_palette)
    folium.GeoJson(
        um_bike_geojson,
        name="Bike network",
        style_function=lambda f, color=test_color_palette: {
            'color': color,
            'weight': 2,
            'opacity': 0.85
        }
    ).add_to(m)
    print(f"I am on number {map_community_double_dex}")

#Non-bike/pedestrian network
# folium.GeoJson(
#     non_lcc_geojson,
#     name="non-connected trails and sidewalks",
#     style_function=lambda f: {
#         'color': '#F50AA9',
#         'weight': 2,
#         'opacity': 0.85
#     }
# ).add_to(m)

try:
    for temp_bbox in used_bboxes:
        folium.Rectangle(
            bounds=[[temp_bbox[0], temp_bbox[1]], [temp_bbox[2], temp_bbox[3]]],
            color='#E24B4A',
            weight=2,
            fill=False
        ).add_to(m)
except:
    print("no used_bboxes found")

# new_bbox = get_new_bbox(bbox, interval, 'northeast')

# folium.Rectangle(
#     bounds=[[new_bbox[0], new_bbox[1]], [new_bbox[2], new_bbox[3]]],
#     color='#E24B4A',
#     weight=2,
#     fill=False
# ).add_to(m)


# Zoom to the network
try:
    bounds = lcc_merged.bounds  # (minx, miny, maxx, maxy)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
except: 
    print("No bounds found")

folium.LayerControl().add_to(m)

#m

NameError: name 'community_geojson_dict' is not defined

# K-means

Doesn't give regard to connections between routes, just geographic location of nodes that said routes are on.

In [84]:
latlon_dict = dict() 

for way in lcc_ways: 
    for node in way.nodes: 
        latlon_dict[node.id] = (node.lat, node.lon)

latlon_node_labels = [key for key in latlon_dict.keys()]
latlon_node_latlons = [latlon_dict[key] for key in latlon_dict.keys()]

kmeans = KMeans(n_clusters = 4, random_state=0, n_init="auto").fit(latlon_node_latlons)

kmeans_label_dict = dict()

for kmeans_label_dex in range(len(kmeans.labels_)): 
    kmeans_label = kmeans.labels_[kmeans_label_dex]
    kmeans_node_label = latlon_node_labels[kmeans_label_dex]
    kmeans_label_dict[kmeans_node_label] = kmeans_label

print("Done with the assignment of labels. Now for the ways.")

# Uncomment the stuff below

# community_geojson_dict = dict()

# for lcc_way_dex in range(len(lcc_ways)):
#     #print(f"I am on {lcc_way_dex}/{len(lcc_ways)}")
#     lcc_way = lcc_ways[lcc_way_dex]
#     count_community_dict = dict() 
#     for node in lcc_way.nodes: 
#         node_kmeans_label = kmeans_label_dict[node.id]
#         if node_kmeans_label not in count_community_dict.keys(): 
#             count_community_dict[node_kmeans_label] = 1
#         else: 
#             count_community_dict[node_kmeans_label] += 1
        
#         if count_community_dict[node_kmeans_label] >= 2: 
#             if node_kmeans_label not in community_geojson_dict.keys(): 
#                 community_geojson_dict[node_kmeans_label] = [lcc_way]
#             else: 
#                 community_geojson_dict[node_kmeans_label].append(lcc_way)
#             #print(count_community_dict)
#             break

#print("Done with assignment of ways.")

Done with the assignment of labels. Now for the ways.


# greedy modularity maximization

Uses Clauset-Newman Moore algorithm with low resolution to break up megatrail into small number of communities. 

At resolution =  0.0001, we get 7 communities, with connections consisting of only a sparse number of routes. 

In [28]:
lcc_subgraph = G.subgraph(lcc_nodes)
communities = nx.community.greedy_modularity_communities(lcc_subgraph, resolution=0.0001)
communities = sorted(communities, key = lambda x: -len(x))

In [29]:
# Go through each way
# check each node: what community does it belong to?
# If 2 or more nodes belong to the same community, add that way to the output community. 

community_geojson_dict = dict()

for lcc_way_dex in range(len(lcc_ways)):
    #print(f"I am on {lcc_way_dex}/{len(lcc_ways)}")
    lcc_way = lcc_ways[lcc_way_dex]
    count_community_dict = dict() 
    for node in lcc_way.nodes: 
        for community_dex in range(len(communities)):
            use_community = communities[community_dex]
            if node.id in use_community: 
                if community_dex not in count_community_dict.keys(): 
                    count_community_dict[community_dex] = 1
                else: 
                    count_community_dict[community_dex] += 1
                break
        if count_community_dict[community_dex] >= 2: 
            if community_dex not in community_geojson_dict.keys(): 
                community_geojson_dict[community_dex] = [lcc_way]
            else: 
                community_geojson_dict[community_dex].append(lcc_way)
            break


In [246]:
import folium
import json
from shapely.geometry import mapping

# Serialize merged network to GeoJSON

# Illinois center
m = folium.Map(location=[40.0, -89.2], zoom_start=6, tiles='CartoDB positron')

hulls = []

# Illinois outline
folium.GeoJson(
    "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
    name="Illinois",
    style_function=lambda f: {
        'fillColor': '#E6F1FB',
        'color': '#185FA5',
        'weight': 1.5,
        'fillOpacity': 0.25
    } if f['properties']['name'] == 'Illinois' else {
        'fillOpacity': 0,
        'color': 'none',
        'weight': 0
    }
).add_to(m)

# Bike/pedestrian network

color_palette = ["#9656a2",'#369acc','#95cf92',"#f8e16f", "#f4895f", "#de324c","#6c584c"]

hull_and_community_dict = dict()

for map_community_double_dex in range(len(list(community_geojson_dict.keys()))):
    hull_and_community = dict()
    #map_community_dex = list(community_geojson_dict.keys())[map_community_double_dex]
    map_community_dex=map_community_double_dex
    use_map_community = community_geojson_dict[map_community_dex]
    use_map_community_merged = convert_to_dispersed_borders(use_map_community)

    #hull_and_community['community'] = use_map_community_merged
    
    #use_map_community_merged = get_outer_bit(use_map_community_merged)
    #use_map_community_merged = use_map_community_merged.simplify(tolerance=0.0001, preserve_topology=True)
    um_bike_geojson = mapping(use_map_community_merged)
    test_color_palette = color_palette[map_community_double_dex % len(color_palette)]
    print(test_color_palette)
    folium.GeoJson(
        um_bike_geojson,
        name="Bike network",
        style_function=lambda f, color=test_color_palette: {
            'color': color,
            'weight': 2,
            'opacity': 1
        }
    ).add_to(m)
    outer_shape = use_map_community_merged.convex_hull

    #hull_and_community['hull'] = outer_shape
    #hull_and_community_dict[map_community_double_dex] = hull_and_community
    
    #hulls.append(outer_shape)
    # outer_shape = mapping(outer_shape)
    # test_color_palette = color_palette[map_community_double_dex % len(color_palette)]
    # folium.GeoJson(
    #     outer_shape,
    #     name="Bike network",
    #     style_function=lambda f, color=test_color_palette: {
    #         'color': color,
    #         'weight': 2,
    #         'opacity': 0.5
    #     }
    # ).add_to(m)
    # print(f"I am on number {map_community_double_dex}")

#Non-bike/pedestrian network

folium.GeoJson(
    non_lcc_geojson,
    name="non-connected trails and sidewalks",
    style_function=lambda f: {
        'color': '#F50AA9',
        'weight': 2,
        'opacity': 0.85
    }
).add_to(m)

# try:
#     for temp_bbox in used_bboxes:
#         folium.Rectangle(
#             bounds=[[temp_bbox[0], temp_bbox[1]], [temp_bbox[2], temp_bbox[3]]],
#             color='#E24B4A',
#             weight=2,
#             fill=False
#         ).add_to(m)
# except:
#     print("no used_bboxes found")

# new_bbox = get_new_bbox(bbox, interval, 'northeast')

# folium.Rectangle(
#     bounds=[[new_bbox[0], new_bbox[1]], [new_bbox[2], new_bbox[3]]],
#     color='#E24B4A',
#     weight=2,
#     fill=False
# ).add_to(m)


# Zoom to the network
try:
    bounds = lcc_merged.bounds  # (minx, miny, maxx, maxy)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
except: 
    print("No bounds found")

folium.LayerControl().add_to(m)



#9656a2
#369acc
#95cf92
#f8e16f
#f4895f
#de324c
#6c584c


In [247]:
#Uncomment this to see the map calculated above
#m

In [241]:
import haversine as hs   
from haversine import Unit
def calc_distance(mlat, mlon, plat, plon): 
    loc1 = (float(mlat), float(mlon))
    loc2 = (float(plat), float(plon))
    distance = hs.haversine(loc1, loc2, unit=Unit.KILOMETERS)
    return distance